# 🌱 AgriSmart — Module 1: Tunisian Arabic Agricultural Advisor

**Pipeline:** Tunisian voice → Whisper ASR → AraT5v2 → Agricultural advice in Tunisian  
**Training:** LoRA fine-tuning on Kaggle 2x T4 (16 GB VRAM total)  
**Stage 1:** AraT5v2 + tunis-ai OR linagora corpus → Tunisian fluency (auto-detected)  
**Stage 2:** Same model + agricultural Q&A pairs → domain expertise

---
### ⚙️ Kaggle Setup
- Runtime: GPU T4 x2
- Internet: **ON** (required for model + dataset download)
- Persistence: Save outputs to `/kaggle/working/`
---

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3
!pip install -q datasets==2.18.0 evaluate==0.4.1
!pip install -q sacrebleu rouge-score sentencepiece
!pip install -q openai-whisper
!pip install -q sacrebleu[ja]
!pip install -q wandb
print('✅ Dependencies installed')

## Cell 2 — Imports & Device Setup

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, load_dataset
import evaluate as hf_evaluate
from sklearn.model_selection import train_test_split

# Kaggle 2x T4 setup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count()
print(f'Device : {DEVICE}')
print(f'GPUs   : {N_GPUS}')
for i in range(N_GPUS):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {mem:.1f} GB')

OUTPUT_DIR = '/kaggle/working/agrismart_nlp'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output : {OUTPUT_DIR}')
print('✅ Imports ready')

## Cell 2b — W&B Initialization

> Get your free API key at https://wandb.ai → Settings → API keys  
> Paste it below once.

In [ ]:
import wandb

WANDB_API_KEY  = ''   # ← Paste your W&B API key here
WANDB_PROJECT  = 'agrismart-nlp'
WANDB_RUN_NAME = 'arat5v2-lora-r16-tunisian'

if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('API key loaded from Kaggle secrets')
    except:
        print('No API key — W&B disabled')
        wandb.init(mode='disabled')
        WANDB_API_KEY = None

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            'model':          'UBC-NLP/AraT5v2-base-1024',
            'lora_r':         16,
            'lora_alpha':     32,
            'lora_dropout':   0.05,
            'lora_targets':   ['q', 'v', 'k', 'o'],
            'max_input':      128,
            'max_output':     256,
            'stage1_lr':      3e-4,
            'stage2_lr':      1e-4,
            'stage1_epochs':  3,
            'stage2_epochs':  10,
            'batch_size':     16,
            'primary_metric': 'chrF',
        }
    )
    print(f'W&B run started: {wandb.run.url}')

## Cell 3 — Agricultural Q&A Dataset (Hand-Crafted)

Tunisian farmer questions → Tunisian agricultural advice.  
**Add more pairs here as you collect them — even 50 more pairs helps.**

In [ ]:
AGRI_QA = [
    # ── Crop Disease ──────────────────────────────────────────
    ('ورق الطماطم عندي بدأ يصفر، شنوة المشكلة؟',
     'اصفرار الورق في الطماطم غالباً سببو نقص الحديد أو الأزوت. حاول تضيف سماد متوازن وتراقب الري. كان الاصفرار يبدأ من الورق القديم، المشكلة في الأزوت. كان يبدأ من الورق الجديد، المشكلة في الحديد أو المنغنيز.'),
    ('نباتات الفلفل متاعي فيها بقع بيضاء على الورق، شنوة هذا؟',
     'البقع البيضاء على الفلفل غالباً تكون البياض الدقيقي. هذا مرض فطري يجي مع الرطوبة العالية. العلاج يكون برش مبيد فطري على بازو كبريت. خلي مسافة بين النباتات باش تتهوى.'),
    ('أوراق الزيتون متاعي بدات تتحول للون البني وتسقط، شنوة نعمل؟',
     'هذا قد يكون مرض عين الطاووس، وهو مرض فطري شائع في الزيتون. الحل هو رش مبيد نحاسي مرتين في السنة، مرة في الخريف ومرة في الربيع. تأكد من تقليم الشجرة باش تتهوى أكثر.'),
    ('شجرة التفاح متاعي فيها ثمار صغيرة ومشوهة، علاش؟',
     'الثمار الصغيرة والمشوهة في التفاح غالباً سببها حشرة السوسة أو نقص في التلقيح. تحقق من وجود حشرات تحت الورق. كان مافيش حشرات، يمكن المشكلة في قلة الري أو نقص البورون في التربة.'),
    ('كيف نعرف كان الطماطم مريضة بالأوبروز؟',
     'مرض الأوبروز في الطماطم تعرفو من الأعراض الجاية: الثمار تبين عليها بقع سوداء أو بنية في الطرف السفلي، الورق يذبل حتى في وجود الماء، والنبات يكبر ببطء. السبب الرئيسي هو نقص الكالسيوم مع ري غير منتظم.'),

    # ── Irrigation ────────────────────────────────────────────
    ('قداش نسقي الطماطم في الصيف؟',
     'في الصيف، الطماطم تحتاج ري كل يوم أو يوم على يوم حسب حرارة الجو والتربة. الأحسن تسقي في الصبح الباكر أو بعد العشية باش تقلل البخر. الري بالتنقيط أحسن من الري بالغمر لأنه يوفر الماء ويقلل الأمراض الفطرية.'),
    ('التربة متاعي تجف بسرعة، شنوة نعمل؟',
     'كان التربة تجف بسرعة، جرب تضيف مادة عضوية مثل الكمبوست باش تحسن قدرة التربة على حفظ الماء. كذلك تغطية سطح التربة بالتبن أو البلاستيك يقلل من البخر بنسبة كبيرة.'),
    ('وين نعرف كان النبتة محتاجة ماء ولا لا؟',
     'علامات احتياج النبتة للماء: الورق يبدأ يذبل في الصباح، لون الورق يتغير لأخضر باهت، والتربة تبين جافة على عمق سنتيمترين. أدخل إصبعك في التربة، كان جافة ارو. كان رطبة انتظر.'),

    # ── Soil & Fertilizer ─────────────────────────────────────
    ('شنوة أحسن سماد للطماطم؟',
     'الطماطم تحتاج سماد متوازن فيه أزوت وفوسفور وبوتاسيوم. في بداية النمو، ركز على الأزوت باش تكبر الورق. وقت الإزهار والثمار، ركز على الفوسفور والبوتاسيوم. السماد العضوي مثل السرجين المتخمر ممتاز لتحسين التربة على المدى الطويل.'),
    ('كيفاش نعرف نوع التربة متاعي؟',
     'تعرف نوع التربة بطرق بسيطة: خذ قبضة تراب ورطبها قليلاً. كان تتشكل كرة ملساء وتلتصق، تربتك طينية. كان تتفتت بسرعة، تربتك رملية. الأحسن تجيب تحليل من مختبر زراعي.'),
    ('كيفاش نحضر الكمبوست في الدار؟',
     'لتحضير الكمبوست: اجمع بقايا الخضار والفواكه والأوراق الجافة. ضعها في طبقات متناوبة بين مواد خضراء ومواد بنية. رطب الخليط بشكل منتظم وقلبه كل أسبوعين. بعد شهرين إلى ثلاثة أشهر يكون الكمبوست جاهز للاستخدام.'),

    # ── Pests ─────────────────────────────────────────────────
    ('كيفاش نتخلص من الحشرات في الفلفل؟',
     'للتخلص من الحشرات في الفلفل: استخدم مبيدات حشرية طبيعية مثل زيت النيم. راقب النباتات بانتظام وخاصة تحت الورق. ضع فخاخ لاصقة صفراء لاصطياد الحشرات الطائرة. في حالة الإصابة الشديدة استشر مختص زراعي.'),
    ('النمل كثير في الحديقة متاعي، شنوة نعمل؟',
     'النمل في الحديقة مشكلة لأنه يحمي حشرات المن ويضر بالنباتات. الحل: ضع حاجز من الرماد أو القرفة حول النباتات. استخدم طعوم النمل الطبيعية. إزالة المن يقلل النمل تلقائياً لأنه مصدر غذائه.'),

    # ── Seasons & Planting ────────────────────────────────────
    ('امتى نزرع الطماطم في تونس؟',
     'في تونس، أحسن وقت لزراعة الطماطم هو من فبراير لأبريل للزراعة الربيعية، ومن يوليو لأغسطس للزراعة الخريفية. تجنب الزراعة في أشد فصول الحر والبرد. ابدأ بالشتلات في البيت قبل الزراعة بأربعة أسابيع.'),
    ('شنوة الخضار اللي تزرع في الشتاء في تونس؟',
     'الخضار المناسبة للشتاء في تونس: الجلبانة، الفول، السبانخ، الخس، الكرنب، القرنبيط، الشمر، والثوم. هذه الخضار تتحمل البرد وتعطي إنتاج جيد من أكتوبر لمارس.'),

    # ── Olive & Fruit Trees ───────────────────────────────────
    ('قداش نسقي الزيتون في الصيف؟',
     'الزيتون البالغ يتحمل الجفاف لكن الري يحسن الإنتاج. في الصيف اسق كل أسبوعين إلى ثلاثة أسابيع حسب نوع التربة. الأشجار الصغيرة تحتاج ري أكثر، كل أسبوع في شدة الحر. تجنب الإفراط في الري لأنه يسبب تعفن الجذور.'),
    ('كيفاش نقلم شجرة الزيتون؟',
     'تقليم الزيتون يكون بعد الحصاد في الشتاء. احذف الأغصان الميتة والمريضة أولاً. ثم خفف التاج باش يدخل الضوء والهواء. لا تحذف أكثر من ثلث الشجرة في مرة واحدة. استخدم أدوات حادة ومعقمة لتجنب العدوى.'),
]

agri_df = pd.DataFrame(AGRI_QA, columns=['input', 'output'])
print(f'Agricultural Q&A pairs: {len(agri_df)}')
print(agri_df.head(2))

## Cell 4 — Load Stage 1 Data from Kaggle Datasets

Loads from 3 Kaggle datasets added in the **Data** tab (no HuggingFace needed):
1. `mksaad/arabic-sentiment-twitter-corpus` → 60k+ Arabic tweets (primary)
2. `naim99/tunisian-texts` → Tunisian dialect corpus (secondary)
3. `waalbannyantudre/tunisian-arabizi-dialect-data-sentiment-analysis` → backup

**Kaggle paths auto-detected** — just add the datasets in the Data tab.

In [ ]:
import os, glob
import pandas as pd

print('Loading Stage 1 corpus from Kaggle datasets...')
print()
tunis_df = None
STAGE1_SOURCE = None

# ── Debug: show everything mounted under /kaggle/input ───────────────────────
print('📂 Mounted datasets:')
for entry in sorted(os.listdir('/kaggle/input')):
    full = os.path.join('/kaggle/input', entry)
    files = glob.glob(os.path.join(full, '**', '*'), recursive=True)
    files = [f for f in files if os.path.isfile(f)]
    print(f'  /kaggle/input/{entry}/ → {len(files)} file(s)')
    for f in files[:5]:
        print(f'      {f}')
print()

# ── Helper: find files by extension anywhere under a root ────────────────────
def find_files(base_path, extensions):
    """Return all files matching extensions under base_path."""
    found = []
    for ext in extensions:
        found += glob.glob(os.path.join(base_path, '**', f'*.{ext}'), recursive=True)
    return sorted(found)

def load_text_col(path):
    """Load a CSV/TSV/XLSX and return a Series of text strings."""
    ext = path.rsplit('.', 1)[-1].lower()
    if ext in ('xlsx', 'xls'):
        df = pd.read_excel(path)
    elif ext == 'tsv':
        df = pd.read_csv(path, sep='\t', encoding='utf-8', on_bad_lines='skip')
    else:
        df = pd.read_csv(path, encoding='utf-8', on_bad_lines='skip')
    print(f'    Columns: {list(df.columns)}')
    # Pick the string column with longest average length
    best_col = None
    best_len = 0
    for c in df.columns:
        if df[c].dtype == object:
            avg = df[c].dropna().astype(str).str.len().mean()
            if avg > best_len:
                best_len = avg
                best_col = c
    if best_col is None:
        best_col = df.columns[0]
    print(f'    Using column: "{best_col}" (avg len {best_len:.0f} chars)')
    texts = df[best_col].dropna().astype(str)
    texts = texts[texts.str.len() > 15].reset_index(drop=True)
    return texts

# ── Attempt 1: Arabic Sentiment Twitter Corpus (TSV files) ───────────────────
# Files: train_Arabic_tweets_positive/negative_*.tsv inside arabic_tweets/
ds1_candidates = find_files('/kaggle/input', ['tsv'])
ds1_twitter = [f for f in ds1_candidates if 'arabic' in f.lower() or 'tweet' in f.lower()]
if ds1_twitter:
    print(f'DS1 — Arabic Twitter TSVs found: {len(ds1_twitter)} file(s)')
    all_texts = []
    for fpath in ds1_twitter:
        try:
            print(f'  Loading: {fpath}')
            t = load_text_col(fpath)
            all_texts.append(t)
            print(f'    Rows: {len(t):,}')
        except Exception as e:
            print(f'    Error: {e}')
    if all_texts:
        combined = pd.concat(all_texts, ignore_index=True).drop_duplicates()
        if len(combined) > 50000:
            combined = combined.sample(50000, random_state=42).reset_index(drop=True)
        tunis_df = pd.DataFrame({'input': combined, 'output': combined})
        STAGE1_SOURCE = 'arabic-twitter'
        print(f'\n✅ Arabic Twitter loaded: {len(tunis_df):,} rows total')
else:
    print('DS1 — No Arabic Twitter TSVs found')

# ── Attempt 2: Tunisian Arabizi (TuniziDataset.csv) ──────────────────────────
if tunis_df is None:
    ds3_candidates = find_files('/kaggle/input', ['csv'])
    tunizi = [f for f in ds3_candidates if 'tunizi' in f.lower() or 'arabizi' in f.lower()]
    if tunizi:
        print(f'DS3 — Tunizi CSV found: {tunizi[0]}')
        try:
            t = load_text_col(tunizi[0])
            tunis_df = pd.DataFrame({'input': t, 'output': t})
            STAGE1_SOURCE = 'arabizi-tunizi'
            print(f'✅ Tunizi loaded: {len(tunis_df):,} rows')
        except Exception as e:
            print(f'DS3 error: {e}')
    else:
        print('DS3 — TuniziDataset.csv not found')

# ── Attempt 3: Naim Mhedhbi XLSX ─────────────────────────────────────────────
if tunis_df is None:
    ds2_candidates = find_files('/kaggle/input', ['xlsx', 'xls'])
    naim = [f for f in ds2_candidates if 'naim' in f.lower() or 'tunisian' in f.lower() or 'mhedhbi' in f.lower()]
    if naim:
        print(f'DS2 — Naim XLSX found: {naim[0]}')
        try:
            t = load_text_col(naim[0])
            tunis_df = pd.DataFrame({'input': t, 'output': t})
            STAGE1_SOURCE = 'naim-tunisian'
            print(f'✅ Naim corpus loaded: {len(tunis_df):,} rows')
        except Exception as e:
            print(f'DS2 error: {e}')
    else:
        print('DS2 — Naim XLSX not found')

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if tunis_df is None:
    print('⚠️  No Stage 1 data loaded — Stage 2 only mode')
    STAGE1_SOURCE = None
else:
    print(f'✅ Stage 1 source : {STAGE1_SOURCE}')
    print(f'   Rows           : {len(tunis_df):,}')
    print(f'   Sample         : {tunis_df["input"].iloc[0][:80]}')


## Cell 5 — Prepare Datasets

Splits Stage 1 and Stage 2 data into train/val. Works for all 3 Kaggle sources.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Stage 1 data prep ────────────────────────────────────────────────────────
if tunis_df is not None:
    # All Kaggle sources are already normalised to input/output in Cell 4
    stage1_df = tunis_df[['input', 'output']].copy()
    # Light quality filter: drop very short or duplicate texts
    stage1_df = stage1_df[stage1_df['input'].str.len() > 20].drop_duplicates(subset='input')
    stage1_df = stage1_df.reset_index(drop=True)
    print(f'Stage 1 ({STAGE1_SOURCE}) — {len(stage1_df):,} samples after filtering')
else:
    stage1_df = None
    print('Stage 1 skipped — Stage 2 only mode')

# ── Stage 2 data prep ────────────────────────────────────────────────────────
stage2_df = agri_df.copy()
print(f'Stage 2 (agricultural Q&A) — {len(stage2_df)} samples')

# ── Splits ────────────────────────────────────────────────────────────────────
if stage1_df is not None:
    s1_train, s1_val = train_test_split(stage1_df, test_size=0.05, random_state=42)
    print(f'Stage 1 — Train: {len(s1_train):,} | Val: {len(s1_val):,}')

s2_train, s2_val = train_test_split(stage2_df, test_size=0.2, random_state=42)
print(f'Stage 2 — Train: {len(s2_train)} | Val: {len(s2_val)}')
print('✅ Datasets ready')


## Cell 6 — Load AraT5v2 Model & Tokenizer

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

MODEL_ID   = 'UBC-NLP/AraT5v2-base-1024'
MAX_INPUT  = 128
MAX_OUTPUT = 256

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f'Loading model...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

total = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {total/1e6:.1f}M parameters')
print(f'Device map: {model.hf_device_map}')

## Cell 7 — Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=['q', 'v', 'k', 'o'],
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,}  ({100*trainable/total:.2f}%)')
print(f'Frozen:    {total-trainable:,}')

## Cell 8 — Tokenization Helper

In [ ]:
def tokenize_batch(df, max_input=MAX_INPUT, max_output=MAX_OUTPUT):
    """Tokenize a DataFrame with input/output columns for Seq2SeqTrainer."""
    hf_ds = Dataset.from_pandas(df[['input', 'output']].reset_index(drop=True))

    def tokenize(examples):
        model_inputs = tokenizer(
            examples['input'],
            max_length=max_input,
            truncation=True,
            padding='max_length',
        )
        labels = tokenizer(
            examples['output'],
            max_length=max_output,
            truncation=True,
            padding='max_length',
        )
        label_ids = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels['input_ids']
        ]
        model_inputs['labels'] = label_ids
        return model_inputs

    return hf_ds.map(tokenize, batched=True, remove_columns=['input', 'output'])

print('✅ Tokenization helper ready')

## Cell 9 — Evaluation Metrics

In [ ]:
bleu_metric  = hf_evaluate.load('sacrebleu')
rouge_metric = hf_evaluate.load('rouge')
chrf_metric  = hf_evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    bleu  = bleu_metric.compute(predictions=decoded_preds,  references=[[l] for l in decoded_labels])
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf  = chrf_metric.compute(predictions=decoded_preds,  references=[[l] for l in decoded_labels])

    return {
        'bleu':   round(bleu['score'],    2),
        'rouge1': round(rouge['rouge1'],  4),
        'rougeL': round(rouge['rougeL'],  4),
        'chrf':   round(chrf['score'],    2),
    }

print('✅ Metrics ready: BLEU + ROUGE + chrF')

## Cell 10 — Stage 1 Training: Tunisian Fluency

Skipped automatically if no Stage 1 data was loaded in Cell 4.

In [ ]:
if stage1_df is None:
    print('⏭️  Skipping Stage 1 — no corpus loaded. Proceeding to Stage 2.')
else:
    print('Tokenizing Stage 1 data...')
    s1_train_ds = tokenize_batch(s1_train)
    s1_val_ds   = tokenize_batch(s1_val)
    s1_train_ds.set_format('torch')
    s1_val_ds.set_format('torch')

    stage1_args = Seq2SeqTrainingArguments(
        output_dir=f'{OUTPUT_DIR}/stage1',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=3e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        fp16=True,
        predict_with_generate=True,
        generation_max_length=MAX_OUTPUT,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='chrf',
        greater_is_better=True,
        report_to='wandb' if WANDB_API_KEY else 'none',
        optim='adamw_torch',
        dataloader_num_workers=2,
    )

    stage1_trainer = Seq2SeqTrainer(
        model=model,
        args=stage1_args,
        train_dataset=s1_train_ds,
        eval_dataset=s1_val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('🚀 Stage 1 training: Tunisian fluency...')
    stage1_trainer.train()
    print('✅ Stage 1 complete!')

    model.save_pretrained(f'{OUTPUT_DIR}/stage1_checkpoint')
    tokenizer.save_pretrained(f'{OUTPUT_DIR}/stage1_checkpoint')
    print('Stage 1 checkpoint saved.')

## Cell 11 — Stage 2 Training: Agricultural Domain

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

print('Tokenizing Stage 2 (agricultural Q&A) data...')
s2_train_ds = tokenize_batch(s2_train)
s2_val_ds   = tokenize_batch(s2_val)
s2_train_ds.set_format('torch')
s2_val_ds.set_format('torch')

stage2_args = Seq2SeqTrainingArguments(
    output_dir=f'{OUTPUT_DIR}/stage2',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=MAX_OUTPUT,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    greater_is_better=True,
    report_to='wandb' if WANDB_API_KEY else 'none',
    optim='adamw_torch',
)

stage2_trainer = Seq2SeqTrainer(
    model=model,
    args=stage2_args,
    train_dataset=s2_train_ds,
    eval_dataset=s2_val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Stage 2 training: Agricultural domain...')
stage2_trainer.train()
print('✅ Stage 2 complete!')

## Cell 12 — Save Final Model

In [ ]:
FINAL_DIR  = f'{OUTPUT_DIR}/final_adapter'
MERGED_DIR = f'{OUTPUT_DIR}/final_merged'

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Adapter saved: {FINAL_DIR}')

merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved: {MERGED_DIR}')
print('Use the merged model for inference.')

## Cell 13 — Evaluation on Agricultural Q&A

In [ ]:
def generate_advice(question: str, max_new_tokens: int = 256) -> str:
    """Generate agricultural advice in Tunisian Arabic."""
    inputs = tokenizer(
        question,
        return_tensors='pt',
        max_length=MAX_INPUT,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = merged.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


preds, refs = [], []
for _, row in s2_val.iterrows():
    pred = generate_advice(row['input'])
    preds.append(pred)
    refs.append(row['output'])

b = bleu_metric.compute(predictions=preds,  references=[[r] for r in refs])
r = rouge_metric.compute(predictions=preds, references=refs)
c = chrf_metric.compute(predictions=preds,  references=[[r] for r in refs])

print('=' * 50)
print('EVALUATION RESULTS (Agricultural Q&A)')
print('=' * 50)
print(f'chrF  : {c["score"]:.2f}  ← primary metric for Arabic')
print(f'BLEU  : {b["score"]:.2f}')
print(f'ROUGE1: {r["rouge1"]:.4f}')
print(f'ROUGEL: {r["rougeL"]:.4f}')
print('=' * 50)

print('\nSample outputs:')
for i in range(min(3, len(s2_val))):
    print(f'\n  Q:    {s2_val["input"].iloc[i]}')
    print(f'  Ref:  {refs[i][:100]}...')
    print(f'  Pred: {preds[i][:100]}...')

## Cell 13b — Log Final Metrics to W&B

In [ ]:
if WANDB_API_KEY and wandb.run:
    wandb.log({
        'final/bleu':   b['score'],
        'final/rouge1': r['rouge1'],
        'final/rougeL': r['rougeL'],
        'final/chrf':   c['score'],
    })

    table = wandb.Table(columns=['Question', 'Reference', 'Prediction', 'chrF'])
    for i in range(min(10, len(s2_val))):
        _c = hf_evaluate.load('chrf')
        _score = _c.compute(predictions=[preds[i]], references=[[refs[i]]])['score']
        table.add_data(
            s2_val['input'].iloc[i],
            refs[i],
            preds[i],
            round(_score, 2)
        )
    wandb.log({'predictions_table': table})
    wandb.finish()
    print('W&B run finished.')
else:
    print('W&B not active — metrics printed above only')

## Cell 14 — Whisper ASR: Voice → Text

Full pipeline: Tunisian voice → Whisper → text → AraT5v2 → agricultural advice

In [ ]:
import whisper

print('Loading Whisper ASR model (small)...')
asr_model = whisper.load_model('small')
print('✅ Whisper ready.')


def transcribe_tunisian(audio_path: str) -> str:
    result = asr_model.transcribe(
        audio_path,
        language='ar',
        task='transcribe',
        fp16=True,
        beam_size=5,
    )
    return result['text'].strip()


def voice_to_advice(audio_path: str) -> dict:
    print('Step 1: Transcribing audio...')
    transcription = transcribe_tunisian(audio_path)
    print(f'  Transcribed: {transcription}')
    print('Step 2: Generating agricultural advice...')
    advice = generate_advice(transcription)
    print(f'  Advice: {advice}')
    return {'transcription': transcription, 'advice': advice}


# ── Text-only test ───────────────────────────────────────────
print('\n--- Text-only pipeline test ---')
test_questions = [
    'ورق الطماطم متاعي يصفر، شنوة المشكلة؟',
    'قداش نسقي الزيتون في الصيف؟',
    'كيفاش نتخلص من الحشرات في الفلفل؟',
]
for q in test_questions:
    print(f'\nQ: {q}')
    print(f'A: {generate_advice(q)}')

## Cell 15 — When MADAR Arrives: How to Add It

Once you receive the MADAR dataset, add it to Stage 1 with this cell.

In [ ]:
MADAR_PATH = '/kaggle/input/madar-corpus/madar_tunisian.tsv'

if os.path.exists(MADAR_PATH):
    madar_df = pd.read_csv(MADAR_PATH, sep='\t')
    print(f'MADAR loaded: {len(madar_df)} rows')
    print(f'Columns: {list(madar_df.columns)}')
    if 'dialect' in madar_df.columns:
        madar_df = madar_df[madar_df['dialect'].str.contains('TUN|aeb|Tunis', case=False)]
    # madar_df = madar_df.rename(columns={'src': 'input', 'tgt': 'output'})
    # combined_stage1 = pd.concat([stage1_df, madar_df[['input','output']]], ignore_index=True)
    print('Adjust column rename above then uncomment the combine line.')
else:
    print('MADAR not found yet. Run after uploading to Kaggle.')

---
## 📋 Summary

| Component | Choice | Reason |
|---|---|---|
| Base model | AraT5v2-base-1024 | Encoder-decoder, Arabic-native, built for translation |
| Fine-tuning | LoRA (r=16, q/v/k/o) | Efficient, preserves base knowledge |
| Stage 1 data | tunis-ai parallel corpus (auto-fallback to linagora) | General Tunisian fluency |
| Stage 2 data | Hand-crafted agricultural Q&A (16 pairs) | Domain expertise |
| Primary metric | chrF | Character-level, robust to Arabic spelling variation |
| ASR | Whisper-small (ar) | Best Arabic accuracy at reasonable speed |
| Inference | Beam search (n=4) | Better quality than greedy for advice generation |

### 🔜 Next steps
1. Expand agricultural Q&A pairs (target: 200+ pairs) in Cell 3
2. Add MADAR when it arrives (Cell 15)
3. Integrate with Module 2 disease detection output